# Week 4 ? Day 4: Hypothesis Testing

## Objective
Use statistical tests to validate business claims with evidence.

We use the sales dataset to demonstrate:
- One-sample t-test
- Two-sample t-test
- P-values and confidence intervals
- Type I and Type II errors


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

df = pd.read_csv("../../week-01/day-05/sales_data.csv")
df.head()


,sale_id,customer_name,product,category,region,amount,sale_date,discount
0,1,Ananya,Laptop,Electronics,South,55000,01-09-2026,0.10
1,2,Rahul,Phone,Electronics,North,25000,02-09-2026,0.05
2,3,Priya,Chair,Furniture,West,4500,03-09-2026,NaN
3,4,Arjun,Desk,Furniture,South,8000,04-09-2026,0.00
4,5,Meera,Headphones,Electronics,East,2000,05-09-2026,0.15


## Scenario 1: One-sample t-test

**Business claim:** The average sale amount is ?10,000.

- Null hypothesis (H?): The population mean is ?10,000.
- Alternative hypothesis (H?): The population mean is different from ?10,000.

We use a one-sample t-test to check the claim.


In [2]:
amounts = df["amount"].dropna()
claimed_mean = 10000

t_stat, p_value = stats.ttest_1samp(amounts, popmean=claimed_mean)
confidence_interval = stats.t.interval(
    0.95,
    df=len(amounts) - 1,
    loc=amounts.mean(),
    scale=stats.sem(amounts)
)

print("Sample mean:", round(amounts.mean(), 2))
print("T-statistic:", round(t_stat, 4))
print("P-value:", round(p_value, 4))
print("95% confidence interval:", confidence_interval)

if p_value < 0.05:
    print("Decision: Reject H0. The average sale differs from ?10,000.")
else:
    print("Decision: Fail to reject H0. There is insufficient evidence of a difference.")


Sample mean: 13120.0
T-statistic: 0.5888
P-value: 0.5705
95% confidence interval: (np.float64(1132.7463992202083), np.float64(25107.25360077979))
Decision: Fail to reject H0. There is insufficient evidence of a difference.


## Scenario 2: Two-sample t-test

**Business question:** Do average sale amounts differ between the North and South regions?

- H?: The two regional means are equal.
- H?: The two regional means are different.

Welch's t-test is used because it does not require equal group variances.


In [3]:
north = df.loc[df["region"] == "North", "amount"].dropna()
south = df.loc[df["region"] == "South", "amount"].dropna()

t_stat_2, p_value_2 = stats.ttest_ind(north, south, equal_var=False)

def welch_confidence_interval(a, b, confidence=0.95):
    mean_diff = a.mean() - b.mean()
    var_a, var_b = a.var(ddof=1), b.var(ddof=1)
    n_a, n_b = len(a), len(b)

    standard_error = np.sqrt(var_a / n_a + var_b / n_b)
    degrees_freedom = (var_a / n_a + var_b / n_b) ** 2 / (
        (var_a / n_a) ** 2 / (n_a - 1) +
        (var_b / n_b) ** 2 / (n_b - 1)
    )
    critical_value = stats.t.ppf((1 + confidence) / 2, degrees_freedom)
    return (mean_diff - critical_value * standard_error,
            mean_diff + critical_value * standard_error)

ci_2 = welch_confidence_interval(north, south)

print("North mean:", round(north.mean(), 2))
print("South mean:", round(south.mean(), 2))
print("T-statistic:", round(t_stat_2, 4))
print("P-value:", round(p_value_2, 4))
print("95% CI for North - South:", ci_2)

if p_value_2 < 0.05:
    print("Decision: Reject H0. Evidence suggests different regional means.")
else:
    print("Decision: Fail to reject H0. Evidence is insufficient to show a difference.")


North mean: 12575.0
South mean: 20375.0
T-statistic: -0.4589
P-value: 0.6796
95% CI for North - South: (np.float64(-64210.134353629204), np.float64(48610.134353629204))
Decision: Fail to reject H0. Evidence is insufficient to show a difference.


## Scenario 3: Discount comparison

**Business question:** Is the average discount different from 10%?

- H?: The average discount is 0.10.
- H?: The average discount is not 0.10.

Rows with missing discounts are excluded.


In [4]:
discounts = df["discount"].dropna()

t_stat_3, p_value_3 = stats.ttest_1samp(discounts, popmean=0.10)
ci_3 = stats.t.interval(
    0.95,
    df=len(discounts) - 1,
    loc=discounts.mean(),
    scale=stats.sem(discounts)
)

print("Average discount:", round(discounts.mean(), 4))
print("T-statistic:", round(t_stat_3, 4))
print("P-value:", round(p_value_3, 4))
print("95% confidence interval:", ci_3)

if p_value_3 < 0.05:
    print("Decision: Reject H0. Average discount differs from 10%.")
else:
    print("Decision: Fail to reject H0. Insufficient evidence of a difference.")


Average discount: 0.0614
T-statistic: -1.8904
P-value: 0.1076
95% confidence interval: (np.float64(0.01150158326884241), np.float64(0.11135555958830046))
Decision: Fail to reject H0. Insufficient evidence of a difference.


## Understanding p-values and significance

- A p-value is the probability of observing results this extreme, assuming H? is true.
- If p < 0.05, the result is commonly called statistically significant and we reject H?.
- If p ? 0.05, we fail to reject H?. This does not prove H? is true.
- A 95% confidence interval gives a range of plausible values for the population parameter.

## Type I and Type II errors

- **Type I error:** Rejecting a true null hypothesis. Business example: concluding a new promotion increases sales when it actually does not. This may waste marketing budget.
- **Type II error:** Failing to reject a false null hypothesis. Business example: missing a real sales improvement from a promotion. This may cause the company to miss an opportunity.

## Conclusion

Hypothesis testing helps analysts make decisions using sample evidence rather than assumptions. Results should be interpreted with business context and sample size in mind.
